In [ ]:
!pip install missingno
!pip install xgboost lightgbm scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import warnings
import gc
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/lhanhduy/43-temp/data_43_temp.csv", skiprows=[1,2]) #trước mắt bỏ 2 dòng đầu cho dễ xử lí
# df = pd.read_csv("/kaggle/input/datasets/anhduy54/43-wind/Data_WindSpeed_43.csv")
print(f"temp_dataset: {df.shape}")

---
# **Create Missing Data**

In [ ]:
def generate_fixed_length_gaps(df_input, gap_days, num_gaps=10, seed=42, rows_per_day=8):
    """
    Tạo bộ dữ liệu missing với CÙNG MỘT độ dài gap cho tất cả các vị trí được chọn.
    Có tích hợp thuật toán chống đè (anti-overlap) để đảm bảo các gap không dính vào nhau.
    """
    df_temp = df_input.copy(deep=True)
    feature_columns = [col for col in df_temp.columns if col != "TimeVN"]
    
    # Đảm bảo format datetime để tìm đúng 1:00 AM
    ts = pd.to_datetime(df_temp['TimeVN'], format='mixed', dayfirst=True, errors='coerce')
    
    # Lấy ra vị trí (integer index) của các dòng có giờ = 1
    # Dùng np.where để lấy vị trí tuyệt đối, an toàn hơn loc khi thao tác
    daily_start_positions = np.where(ts.dt.hour == 1)[0].tolist()
    
    np.random.seed(seed)
    gap_rows = gap_days * rows_per_day
    
    # Dictionary lưu lại các index bị xóa để sau này tính RMSE, MAE
    missing_ground_truth = {col: [] for col in feature_columns}

    for col in feature_columns:
        col_idx = df_temp.columns.get_loc(col)
        
        # Chỉ giữ lại các vị trí bắt đầu mà khi cộng thêm gap_rows không vượt quá chiều dài data
        valid_starts = [pos for pos in daily_start_positions if pos + gap_rows <= len(df_temp)]
        
        available_starts = valid_starts.copy()
        
        for _ in range(num_gaps):
            if not available_starts:
                print(f"Cảnh báo: Không đủ khoảng trống để tạo đủ {num_gaps} gaps cho cột {col}")
                break
                
            # Chọn ngẫu nhiên 1 vị trí bắt đầu
            start_pos = np.random.choice(available_starts)
            end_pos = start_pos + gap_rows
            
            # Xóa dữ liệu (gán NaN)
            df_temp.iloc[start_pos:end_pos, col_idx] = np.nan
            
            # Lưu lại vị trí đã đục lỗ
            missing_ground_truth[col].extend(list(range(start_pos, end_pos)))
            
            # --- CƠ CHẾ CHỐNG ĐÈ (ANTI-OVERLAP) ---
            # Xóa bỏ các vị trí bắt đầu (start_pos) lân cận ra khỏi danh sách available_starts
            # Khoảng cách tối thiểu giữa 2 điểm bắt đầu phải lớn hơn chiều dài của gap
            available_starts = [pos for pos in available_starts if abs(pos - start_pos) > gap_rows]

    return df_temp, missing_ground_truth

# ================= TẠO 4 BỘ DATASET ĐỘC LẬP =================

# Giả sử 'df' là dataframe gốc của bạn
gap_scenarios = [1, 3, 5, 7]
missing_datasets = {}       # Chứa 4 dataframe đã bị đục lỗ
ground_truth_indices = {}   # Chứa vị trí các lỗ hổng để tính sai số sau này

for days in gap_scenarios:
    print(f"Đang tạo dataset cho kịch bản missing {days} ngày liên tục...")
    
    # Gọi hàm cho từng độ dài
    df_miss, truth_dict = generate_fixed_length_gaps(
        df_input=df, 
        gap_days=days, 
        num_gaps=10,   # Tùy chỉnh số lượng đoạn đứt gãy bạn muốn tạo
        seed=42,       # Giữ nguyên seed để kết quả random có thể tái lập được
        rows_per_day=8
    )
    
    # Lưu vào dictionary
    missing_datasets[f'gap_{days}d'] = df_miss
    ground_truth_indices[f'gap_{days}d'] = truth_dict
    
    total_nan = df_miss.isna().sum().sum()
    print(f"-> Hoàn tất! Tổng số NaN tạo ra: {total_nan}\n")

# Để truy xuất data sử dụng:
df_1_day = missing_datasets['gap_1d']
df_3_days = missing_datasets['gap_3d']
df_5_days = missing_datasets['gap_5d']
df_7_days = missing_datasets['gap_7d']

df_missing = df_7_days

In [ ]:
def split_by_hierarchical_correlation(df, low_thresh, high_thresh, time_col='TimeVN'):
    # Tách dữ liệu số để tính toán
    df_numeric = df.select_dtypes(include=[np.number])
    
    # 1. Ensemble Correlation
    corr = (df_numeric.corr(method='pearson').abs() + 
            df_numeric.corr(method='spearman').abs()) / 2
    
    # 2. Distance Matrix & Linkage (Dùng cho Dendrogram)
    dist_matrix = 1 - corr.fillna(0)
    dist_vec = squareform(dist_matrix, checks=False)
    linkage_matrix = hierarchy.ward(dist_vec)
    
    # 3. Tính Degree Centrality (Mức độ quan trọng trung bình của biến trong mạng lưới)
    global_scores = (corr.sum() - 1) / (len(corr) - 1)
    
    # 4. Phân tách theo 3 ngưỡng
    # Low: score < 0.3
    low_cols = global_scores[global_scores < low_thresh].index.tolist()
    # Medium: 0.3 <= score < 0.8
    med_cols = global_scores[(global_scores >= low_thresh) & (global_scores < high_thresh)].index.tolist()
    # High: score >= 0.8
    high_cols = global_scores[global_scores >= high_thresh].index.tolist()
    
    # 5. Trả kết quả (Kèm theo cột thời gian nếu có)
    time_list = [time_col] if time_col in df.columns else []
    
    df_low = df[time_list + low_cols]
    df_med = df[time_list + med_cols]
    df_high = df[time_list + high_cols]
    
    return df_low, df_med, df_high, global_scores, linkage_matrix

# --- Thực thi ---
df_low, df_med, df_high, scores, linkage = split_by_hierarchical_correlation(df_missing, low_thresh=0.6, high_thresh=0.75)

---
# **Models**

In [ ]:
import pandas as pd
import numpy as np
import gc
import os
from tqdm.auto import tqdm
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. HÀM ĐÁNH GIÁ METRICS
# ==========================================
def get_final_metrics(df_orig, df_imp, df_miss):
    stats = []
    common_cols = [c for c in df_miss.columns if c in df_orig.columns and c in df_imp.columns and c not in ['TimeVN', 'Time', 'Date']]
    
    for col in common_cols:
        mask = pd.isna(df_miss[col].values) & ~pd.isna(df_orig[col].values)
        yt, yp = df_orig[col].values[mask], df_imp[col].values[mask]
        valid = ~np.isnan(yp); yt, yp = yt[valid], yp[valid]
        if len(yt) < 2: continue
            
        T = len(yt)
        v_min, v_max = np.nanmin(df_orig[col].values), np.nanmax(df_orig[col].values)
        rng = v_max - v_min if v_max > v_min else 1.0
        
        rmse = np.sqrt(np.mean((yt - yp)**2))
        mae = np.mean(np.abs(yp - yt))
        sim = (1/T) * np.sum(1 / (1 + (np.abs(yp - yt) / rng)))
        nse = 1 - (np.sum((yt - yp)**2) / (np.sum((yt - np.mean(yt))**2) + 1e-9))
        r2 = r2_score(yt, yp)
        
        stats.append({
            "Station": col, "NSE": nse, "R2": r2, "Sim": sim,
            "RMSE": rmse, "MAE": mae, "NMAE": mae / rng
        })
    return pd.DataFrame(stats)

# ==========================================
# 2. HÀM TẠO DỮ LIỆU MỒI BẰNG MEAN (TRUNG BÌNH)
# ==========================================
def get_mean_seed(df_miss):
    # Lấy các cột số
    df_num = df_miss.select_dtypes(include=[np.number]).copy().astype('float32')
    
    # Điền khuyết bằng giá trị trung bình của từng cột
    df_filled = df_num.fillna(df_num.mean())
    
    # Đề phòng trường hợp có cột bị NaN toàn bộ 100%, ta điền 0 để model không bị lỗi
    df_filled = df_filled.fillna(0)
    
    return df_filled

# ==========================================
# 3. CUSTOM MICE FRAMEWORK (Chuẩn truyền thống)
# ==========================================
def run_custom_mice_imputation(df_miss, df_seed, estimator_model, max_iter=5, tol=1e-3):
    df_num = df_miss.select_dtypes(include=[np.number]).astype('float32')
    target_cols = df_num.columns.tolist()
    working = df_seed[target_cols].copy()
    
    for iteration in range(max_iter):
        old_working = working.copy()
        for target in target_cols:
            m_idx = df_num[target].isna()
            if not m_idx.any(): continue
            
            pred_cols = [c for c in target_cols if c != target]
            X_train = working.loc[~m_idx, pred_cols].values
            X_test = working.loc[m_idx, pred_cols].values
            y_train = df_num.loc[~m_idx, target].values
            
            estimator_model.fit(X_train, y_train)
            working.loc[m_idx, target] = estimator_model.predict(X_test)
            
        # Sửa lại cách tính diff an toàn hơn khi dùng numpy array trực tiếp
        diff = np.sqrt(np.mean((old_working.values - working.values)**2))
        if diff < tol: 
            # print(f"Hội tụ sớm tại vòng lặp thứ {iteration + 1}")
            break 
            
    return working

# ==========================================
# 4. KỊCH BẢN CHẠY & LƯU KẾT QUẢ
# ==========================================

# 🔴 CHỈNH SỬA Ở ĐÂY CHO MỖI LẦN CHẠY 🔴
METHOD_NAME = "MEAN_MICE"  # Đổi tên thành MEAN_MICE để phân biệt với bản SICE của bạn
LEAD_TIME = "7D"           # Thời gian dự báo
# -------------------------------------------------------------

# Tạo thư mục lưu trữ (vd: Results_MEAN_MICE_7D)
SAVE_DIR = f"Results_{METHOD_NAME}_{LEAD_TIME}"
os.makedirs(SAVE_DIR, exist_ok=True)

mice_variants = {
    'LN': LinearRegression(n_jobs=-1),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    'DT': DecisionTreeRegressor(random_state=42),
    'SVR': SVR(kernel='rbf'), 
    'RF': RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42),
    'GB': GradientBoostingRegressor(n_estimators=50, random_state=42),
    'Ada': AdaBoostRegressor(n_estimators=50, random_state=42),
    'XGB': XGBRegressor(n_estimators=50, n_jobs=-1, random_state=42, verbosity=0),
    'LGBM': LGBMRegressor(n_estimators=50, n_jobs=-1, random_state=42, verbose=-1)
}

# (Lưu ý: Bạn nhớ đảm bảo df, df_low, df_med, df_high đã được định nghĩa trước khi chạy ô này)
missing_datasets = {'LOW': df_low, 'MEDIUM': df_med, 'HIGH': df_high}
mice_results = []

print(f"🧬 Bắt đầu chạy {METHOD_NAME} (Pre-impute: Mean) cho tập {LEAD_TIME}...\n")
pbar_levels = tqdm(missing_datasets.items(), desc="📊 LEVEL", colour='magenta', position=0)

for level_name, df_group_missing in pbar_levels:
    
    # 1. Chạy bước khởi tạo mồi bằng MEAN (Rất nhanh)
    df_seed_mean = get_mean_seed(df_group_missing)
    
    pbar_mice = tqdm(mice_variants.items(), desc=f"🏆 MODEL ({level_name})", colour='green', position=1, leave=False)
    for mice_name, estimator in pbar_mice:
        
        # 2. Chạy Imputation (Lặp max 5 vòng)
        df_imp = run_custom_mice_imputation(
            df_miss=df_group_missing, 
            df_seed=df_seed_mean, 
            estimator_model=estimator,
            max_iter=5
        )
        
        # 3. Đánh giá Metrics
        metrics_df = get_final_metrics(df, df_imp, df_group_missing)
        
        if not metrics_df.empty:
            s = metrics_df.mean(numeric_only=True)
            mice_results.append({
                'Level': level_name,
                'Model': mice_name,
                'NSE (↑)': s['NSE'], 'R2 (↑)': s['R2'], 'Sim (↑)': s['Sim'],
                'RMSE (↓)': s['RMSE'], 'MAE (↓)': s['MAE'], 'NMAE (↓)': s['NMAE']
            })
        
        # 4. LƯU BẢNG DATA SAU KHI ĐIỀN KHUYẾT
        df_imp_save = df_imp.copy()
        if 'TimeVN' in df_group_missing.columns:
            df_imp_save.insert(0, 'TimeVN', df_group_missing['TimeVN'].values)
            
        # Đặt tên file: VD: Data_MEAN_MICE_7D_LOW_LGBM.csv
        imputed_filename = f"Data_{METHOD_NAME}_{LEAD_TIME}_{level_name}_{mice_name}.csv"
        save_path = os.path.join(SAVE_DIR, imputed_filename)
        df_imp_save.to_csv(save_path, index=False)

        # Xoá để giải phóng RAM
        del df_imp, df_imp_save
        gc.collect()

# ==========================================
# 5. XUẤT KẾT QUẢ METRICS & HIỂN THỊ
# ==========================================
df_mice_results = pd.DataFrame(mice_results)
df_mice_results['Level'] = pd.Categorical(df_mice_results['Level'], categories=['LOW', 'MEDIUM', 'HIGH'], ordered=True)
df_mice_results = df_mice_results.sort_values(by=['Level', 'Model']).set_index(['Level', 'Model'])

# Lưu file Metrics tổng hợp
metrics_filename = f"Metrics_{METHOD_NAME}_{LEAD_TIME}.csv"
df_mice_results.to_csv(os.path.join(SAVE_DIR, metrics_filename))

print(f"\n✅ Hoàn tất! Toàn bộ file đã được lưu gọn gàng trong thư mục: '{SAVE_DIR}'")

display(df_mice_results.style.format(precision=4)\
    .background_gradient(cmap='RdYlGn', subset=['NSE (↑)', 'R2 (↑)'])\
    .background_gradient(cmap='RdYlGn_r', subset=['RMSE (↓)', 'MAE (↓)', 'NMAE (↓)']))